In [1]:
import cobra
import pandas as pd 

from Bio.Seq import Seq
from Bio.Alphabet import generic_rna

import numpy as np
import statsmodels.api as sm
import scipy.stats as st

import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *
import gene_information as gi

load environmental variables
The project root is: /Users/joycebaghdassarian/Documents/UCSD/Lewis_Lab/Projects/human_me/


In [2]:
psim_me = pd.read_csv(local_data_path + 'processed/psim_me.csv', index_col = 0)
human_model = cobra.io.load_json_model(local_data_path + 'processed/corrected_recon2_2.json')
sp_dict = {1: True, 0: False}
ptm_cols = ['DSB', 'GPI', 'NG', 'OG']
ptm_keys = list(gi.allowed_ptms.keys())

In [3]:
gene1_id = human_model.genes[0].id

idx  = psim_me[psim_me['HGNC_ID'] == gene1_id].index
ptms_ = dict(zip(ptm_keys, psim_me.loc[idx, ptm_cols].iloc[0,:].tolist()))
ptms_ = {k:v for k,v in ptms_.items() if v != 0 and not pd.isna(v)}
fl = psim_me.loc[idx, 'Location'].tolist()[0]

pm,m,p = psim_me.loc[idx, 'PREMRNA_SEQ'].tolist()[0], psim_me.loc[idx, 'MRNA_SEQ'].tolist()[0], psim_me.loc[idx, 'PROTEIN_SEQ'].tolist()[0]
sp = sp_dict[psim_me.loc[idx, 'SP'].tolist()[0]]

# initialize the gene class
gene1 = gi.gene_information(human_model, hgnc_id = gene1_id, ptms = ptms_, 
                         tmd = psim_me.loc[idx,'TMD'].tolist()[0], sp = sp_dict[psim_me.loc[idx,'SP'].tolist()[0]], 
                        keff = None)
gene1.get_final_locations(metabolic_model = human_model, final_locations=fl)

gene1.get_sequences(premrna_seq=pm, mrna_seq=m, protein_seq=p)
gene1.check_gene_information()

print(gene1.module)
print(gene1.hgnc_id)
print(gene1.sp)
print(gene1.ptms)
print(gene1.tmd)
print(gene1.final_locations)
print(gene1.polyA_length)

No errors raised
Machinery
HGNC:550
False
{'dsb': 4.0, 'ng': 6.0, 'og': 1.0}
1.0
{'c': 'Cytosolic Tranport'}
nan


../scripts/gene_information.py:113 UserWarning: No keff specified for this enzyme, will assume a value in model building
../scripts/gene_information.py:132 UserWarning: Final location extacted from cobrapy model, will disregard user input.
../scripts/gene_information.py:245 UserWarning: PTMs are not considered for machinery proteins currently


# start of actual script

In [4]:
# polyA params
polyA = pd.read_csv(local_data_path + 'processed/polyA_length.csv', index_col = 0)
params = st.johnsonsu.fit(polyA.MEAN)
norm_sd = int(polyA.MEAN.min()/3)
idx = sorted(set(polyA.SD.dropna().index.tolist()).intersection(polyA.MEAN.dropna().index.tolist()))
reg_data = polyA.loc[idx, ['SD', 'MEAN']]
X = sm.add_constant(reg_data.MEAN, prepend = False)
mod = sm.OLS(reg_data.SD, X).fit()
min_polyA_mean = -mod.params['const']/mod.params['MEAN']

# L_polyA_n = 250 #https://www.nature.com/articles/s41592-019-0503-y

In [5]:
# mod.summary()

In [6]:
human_model = cobra.io.load_json_model(local_data_path + 'processed/corrected_recon2_2.json')

In [7]:
atp_n = human_model.metabolites.get_by_id('atp[n]')
gtp_n = human_model.metabolites.get_by_id('gtp[n]')

seq_metabolite_map = {human_model.metabolites.get_by_id('utp[n]'): 'U' , 
                      gtp_n: 'G',
                      human_model.metabolites.get_by_id('ctp[n]'): 'C',
                      atp_n: 'A'}

seq_element_map = dict()

for k,v in seq_metabolite_map.items():
    elements = k.elements
    elements['O'] = elements['O'] - 7 # lost from incoming ntp
    elements['P'] = elements['P'] - 2 # lost from incoming ntp
    elements['H'] = elements['H'] - 1 # lost from 3' end of growing strand
    seq_element_map[v] = elements
ppi_n = human_model.metabolites.get_by_id('ppi[n]')


# machinery

# elongation machinery
rnap2 = pd.read_csv(local_data_path + 'raw/RNAP2_HUGO.csv', index_col = None, skiprows = [0])
tfiis, tfiif, ell = ['HGNC:11612', 'HGNC:11614'], ['HGNC:4652', 'HGNC:4653'], ['HGNC:23114', 'HGNC:17064', 'HGNC:23113']
elongin = pd.read_csv(local_data_path + 'raw/elongin.csv', index_col = None, skiprows = [0])
elongator = pd.read_csv(local_data_path + 'raw/elongator.csv', index_col = None, skiprows = [0])
fact = ['HGNC:11327', 'HGNC:11465']
ec = rnap2['HGNC ID (gene)'].tolist() + elongin['HGNC ID (gene)'].tolist() + elongator['HGNC ID (gene)'].tolist()
ec += tfiis + tfiif + ell + fact

# processing machinery and variables
# L_polyA_n = 250 # https://www.nature.com/articles/s41592-019-0503-y
pi_n = human_model.metabolites.get_by_id('pi[n]')
h_n = human_model.metabolites.get_by_id('h[n]')
h2o_n = human_model.metabolites.get_by_id('h2o[n]')
gp = gtp_n.elements
gp['O'] -= 6
gp['P'] -= 2
amet_n = human_model.metabolites.get_by_id('amet[n]')
ahcys_n = human_model.metabolites.get_by_id('ahcys[n]')

cpsf = ['HGNC:2324', 'HGNC:2327', 'HGNC:19124', 'HGNC:2325', 'HGNC:2326', 'HGNC:25651', 'HGNC:13871']
cstf = ['HGNC:2483', 'HGNC:2484', 'HGNC:2485']
cfim, cfiim = ['HGNC:14981', 'HGNC:15970', 'HGNC:14982'], ['HGNC:30097', 'HGNC:16999']
polyA = cpsf + cstf + cfim + cfiim

nelf = ['HGNC:12768', 'HGNC:24324', 'HGNC:15934', 'HGNC:13974']
capping = nelf + ['HGNC:10073', 'HGNC:10075', 'HGNC:21077', 'HGNC:7658', 'HGNC:7659', 'HGNC:11467', 'HGNC:11469',
                 'HGNC:29200', 'HGNC:17970']

In [15]:
class Transcript():
    def __init__(self, gene_information):
        '''Input is an object of the gene_information class, output is all the information needed to 
        build the transcription reations.'''
        self.premrna_seq = Seq(gene_information.premrna_seq, generic_rna)
        self.mrna_seq = Seq(gene_information.mrna_seq, generic_rna) 
        self.id = gene_information.hgnc_id
        
        self.premrna_base_counts, mrna_base_counts = dict(), dict()
        for base_letter in seq_element_map.keys():
            self.premrna_base_counts[base_letter] = self.premrna_seq.count(base_letter)
            mrna_base_counts[base_letter] = self.mrna_seq.count(base_letter)
        
        
        if pd.isna(gene_information.polyA_length):
            self.polyA_length = round(st.johnsonsu.rvs(loc=params[-2], scale=params[-1], *params[:-2]))
        else:
            if gene_information.polyA_length > min_polyA_mean:
                self.polyA_length = round(mod.predict((gene_information.polyA_length, 1)))
            else:
                self.polyA_length = round(gene_information.polyA_length)
        
        # metabolite output of transcriptional elongation and processing reactions----------------------
        # combined to save on compute time/for loops
        self.elongated_transcript, self.processed_transcript = cobra.Metabolite(self.id + '_elongated_transcript[n]'), cobra.Metabolite(self.id + '_processed_transcript[n]')
        self.elongated_transcript.compartment, self.processed_transcript.compartment = 'n', 'n'
        
        elongated_elements, processed_elements = {'C': 0, 'H': 0, 'N': 0, 'O': 0, 'P': 0}, {'C': 0, 'H': 0, 'N': 0, 'O': 0, 'P': 0}
        for base_letter in seq_element_map.keys():
            for element in elongated_elements.keys():
                elongated_elements[element] += self.premrna_base_counts[base_letter]*seq_element_map[base_letter][element]
                processed_elements[element] += (mrna_base_counts[base_letter]*seq_element_map[base_letter][element]) 
        
        #3 and 5' ends
        for dict_ in [elongated_elements, processed_elements]:
            dict_['P'] += 2
            dict_['O'] += 7
            dict_['H'] += 1
        
        self.elongated_transcript.elements = elongated_elements
        self.elongated_transcript.charge = -len(self.premrna_seq) - 3 # -3 for 5' end triphosphate
        
        ### procssed specific
        
        for element in processed_elements.keys():
            processed_elements[element] += (self.polyA_length*seq_element_map['A'][element]) # polyA tail
            processed_elements[element] += gp[element] #5' cap rxn2 - addition of Gp
        
        # 5' cap 
        processed_elements['P'] -= 1 # rxn 1: lost of third triphosphate by RTPase
        processed_elements['O'] -= 4 # rxn 1: loss of third triophosphate by RTPase
        processed_elements['C'] += 2 # rxn 3-4: methyltransferase - cap0 and cap1 structure
        processed_elements['H'] += 5 # methyltransferase - cap0 and cap1 structure
        
        self.processed_transcript.elements = processed_elements
        
        # -3 as for elongated_transcript, +2 for cap
        self.processed_transcript.charge = -len(self.mrna_seq) - 3 - self.polyA_length + 2
        
        
        self.lariat_base_counts = {k: v - mrna_base_counts[k] for k,v in self.premrna_base_counts.items()}
        
    def build_transcript_elongation_reaction(self):
        '''Input is an object of the Transcript class. Output is reaction (cobra.Reaction object) for
        transcriptional elongation of that gene.'''
        
        # elongation reaction
        # https://www.google.com/search?q=rna+polymerization+reaction&source=lnms&tbm=isch&sa=X&ved=2ahUKEwiN_73Vk7rqAhXOsJ4KHW5lB4UQ_AUoAXoECA4QAw&biw=1920&bih=1001#imgrc=w7XH4mHmJglCuM
        self.transcript_elongation = cobra.Reaction(self.id + '_transcription_elongation')
        self.transcript_elongation.subsytem = 'Transcription'
        
        rxn = dict()
        for ntp, base_letter in seq_metabolite_map.items():
            rxn[ntp] = -1*self.premrna_base_counts[base_letter]
        # pyrophosphate released per base added, -1 for 3/5' ends
        rxn[ppi_n] = len(self.premrna_seq) - 1
        rxn[self.elongated_transcript] = 1
        # ATP consumption due to PTMs of nucleosomes
        # https://www.pnas.org/content/pnas/suppl/2015/10/29/1514974112.DCSupplemental/pnas.1514974112.sapp.pdf
        # can perhaps add later

        self.transcript_elongation.add_metabolites(rxn)
        self.transcript_elongation.gene_reaction_rule = ' and '.join(ec) # GPRs
                
    def build_transcript_processing_reaction(self):
        '''Processing includes capping, splicing, and polyA tail.'''
        
        # combine in to one to not create too many reactions
        # capping itself is 4 reactions
        
        self.transcript_processing = cobra.Reaction(self.id + '_transcription_processing')
        self.transcript_processing.subsytem = 'Transcription'
        rxn = dict()
        rxn[atp_n], rxn[ppi_n] = -self.polyA_length, self.polyA_length # polyA tail 
        
        # 5' cap: https://sites.google.com/site/learnorganicchem/organic-molecules/biomolecules/rna/rna-processing?tmpl=%2Fsystem%2Fapp%2Ftemplates%2Fprint%2F&showPrintDialog=1
        rxn[h2o_n], rxn[pi_n] = -1, 1 #rtpase
        rxn[gtp_n] = -1 #gp transfer
        rxn[ppi_n] += 1 # gp transfer
        rxn[amet_n], rxn[ahcys_n] = -2, 2 # methyltransferase - cap0 and cap1 structure
        rxn[h_n] = 1 # methyltransferase cap1
        
        rxn[self.elongated_transcript] = -1
        rxn[self.processed_transcript] = 1
        
        self.transcript_processing.add_metabolites(rxn)
        self.transcript_elongation.gene_reaction_rule = ' and '.join(polyA + capping) # GPRs

In [16]:
self = Transcript(gene1)
self.build_transcript_elongation_reaction()
self.build_transcript_processing_reaction()

In [17]:
self.lariat_base_counts

{'U': 839, 'G': 779, 'C': 726, 'A': 592}

In [18]:
self.polyA_length

108.0

In [19]:
gene1.polyA_length

nan